### Import libraries
Spark functions for column operations; DeltaTable for merging into existing tables.

In [0]:
%python
from pyspark.sql import functions as F
from delta.tables import DeltaTable

### Load utilities
Brings in shared schema names, S3 path and the table name helper.

In [0]:
%run ../../01_setup/utilities

### Utilities
Shared settings (catalog, schemas, S3 path) and a helper for full table names. Loaded by every pipeline notebook with `%run`.

### Notebook parameters
Widgets make the notebook reusable: catalog and data source can be changed without editing code.

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "products", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f"{s3_base_path}/{data_source}/*.csv"
print(base_path)

s3://sportsbar-data-lake-vkt/products/*.csv


## Bronze

### Read raw CSVs from S3
Loads the files as-is, adding load time, source file name and file size for traceability.

In [0]:
df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(base_path)
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)

df.printSchema()
display(df.limit(10))

root
 |-- product_name: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



product_name,product_id,category,read_timestamp,file_name,file_size
SportsBar Energy Bar Choco Fudge (60g),25891101,energy bars,2026-11-04T06:45:27.948Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (40g),25891102,energy bars,2026-11-04T06:45:27.948Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (25g),25891103,energy bars,2026-11-04T06:45:27.948Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (45g),25891201,protien bars,2026-11-04T06:45:27.948Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (55g),25891202,protien bars,2026-11-04T06:45:27.948Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (65g),25891203,protien bars,2026-11-04T06:45:27.948Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (400g),25891301,granola & cereals,2026-11-04T06:45:27.948Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (300g),25891302,granola & cereals,2026-11-04T06:45:27.948Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (200g),25891303,granola & cereals,2026-11-04T06:45:27.948Z,products.csv,1388
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,recovery dairy,2026-11-04T06:45:27.948Z,products.csv,1388


### Save to bronze
Stores the raw data as a Delta table with Change Data Feed on.

In [0]:
(
    df.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .mode("overwrite")
    .saveAsTable(table_name(bronze_schema, data_source))
)

## Silver

### Load bronze data

In [0]:
df_bronze = spark.table(table_name(bronze_schema, data_source))
df_bronze.printSchema()
display(df_bronze)

root
 |-- product_name: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)



product_name,product_id,category,read_timestamp,file_name,file_size
SportsBar Energy Bar Choco Fudge (60g),25891101,energy bars,2026-11-04T06:45:31.868Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (40g),25891102,energy bars,2026-11-04T06:45:31.868Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (25g),25891103,energy bars,2026-11-04T06:45:31.868Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (45g),25891201,protien bars,2026-11-04T06:45:31.868Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (55g),25891202,protien bars,2026-11-04T06:45:31.868Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (65g),25891203,protien bars,2026-11-04T06:45:31.868Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (400g),25891301,granola & cereals,2026-11-04T06:45:31.868Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (300g),25891302,granola & cereals,2026-11-04T06:45:31.868Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (200g),25891303,granola & cereals,2026-11-04T06:45:31.868Z,products.csv,1388
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,recovery dairy,2026-11-04T06:45:31.868Z,products.csv,1388


### 1. Remove duplicate products
Keeps one row per product ID.

In [0]:
display(df_bronze.groupBy("product_id").count().filter(F.col("count") > 1))

print("Rows before:", df_bronze.count())
df_silver = df_bronze.dropDuplicates(["product_id"])
print("Rows after:", df_silver.count())

product_id,count
25891101,2
25891102,2


Rows before: 20
Rows after: 18


In [0]:
dup_ids = df_bronze.groupBy("product_id").count().filter(F.col("count") > 1).select("product_id")
display(df_bronze.join(dup_ids, "product_id").orderBy("product_id"))

product_id,product_name,category,read_timestamp,file_name,file_size
25891101,SportsBar Energy Bar Choco Fudge (60g),energy bars,2026-11-04T06:45:31.868Z,products.csv,1388
25891101,SportsBar Energy Bar Choco Fudge (60g),energy bars,2026-11-04T06:45:31.868Z,products.csv,1388
25891102,SportsBar Energy Bar Choco Fudge (40g),energy bars,2026-11-04T06:45:31.868Z,products.csv,1388
25891102,SportsBar Energy Bar Choco Fudge (40g),energy bars,2026-11-04T06:45:31.868Z,products.csv,1388


### 2. Trim spaces and fix capitalisation in category
"energy bars " → "Energy Bars".

In [0]:
df_silver.select("category").distinct().show()

df_silver = (
    df_silver
    .withColumn("product_name", F.trim(F.col("product_name")))
    .withColumn("category", F.initcap(F.trim(F.col("category"))))
)

df_silver.select("category").distinct().show()

+-----------------+
|         category|
+-----------------+
|      energy bars|
|     protien bars|
|granola & cereals|
|   recovery dairy|
|   healthy snacks|
|  electrolyte mix|
+-----------------+

+-----------------+
|         category|
+-----------------+
|      Energy Bars|
|     Protien Bars|
|Granola & Cereals|
|   Recovery Dairy|
|   Healthy Snacks|
|  Electrolyte Mix|
+-----------------+



### 3. Fix the "Protien" spelling mistake
Replaces "Protien" with "Protein" in product names and categories, whatever the capitalisation.

In [0]:
df_silver = (
    df_silver
    .withColumn("product_name", F.regexp_replace("product_name", "(?i)protien", "Protein"))
    .withColumn("category", F.regexp_replace("category", "(?i)protien", "Protein"))
)

df_silver.select("category").distinct().show()

+-----------------+
|         category|
+-----------------+
|      Energy Bars|
|     Protein Bars|
|Granola & Cereals|
|   Recovery Dairy|
|   Healthy Snacks|
|  Electrolyte Mix|
+-----------------+



### 4. Add division
SportsBar has no division, so it is derived from category to match Atlikon's product hierarchy.

In [0]:
df_silver = df_silver.withColumn(
    "division",
    F.when(F.col("category").isin("Energy Bars", "Protein Bars"), "Nutrition Bars")
     .when(F.col("category") == "Granola & Cereals", "Breakfast Foods")
     .when(F.col("category") == "Recovery Dairy", "Dairy & Recovery")
     .when(F.col("category") == "Healthy Snacks", "Healthy Snacks")
     .when(F.col("category") == "Electrolyte Mix", "Hydration & Electrolytes")
     .otherwise("Other")
)

# Check: should list no "Other" rows
display(df_silver.groupBy("division", "category").count().orderBy("division"))

division,category,count
Breakfast Foods,Granola & Cereals,3
Dairy & Recovery,Recovery Dairy,3
Healthy Snacks,Healthy Snacks,3
Hydration & Electrolytes,Electrolyte Mix,3
Nutrition Bars,Energy Bars,3
Nutrition Bars,Protein Bars,3


### 5. Extract variant from product name
Takes the text in brackets, e.g. "Choco Bar (60g)" → "60g". Empty if there are no brackets.

In [0]:
df_silver = df_silver.withColumn(
    "variant",
    F.regexp_extract(F.col("product_name"), r"\((.*?)\)", 1)
)

display(df_silver.select("product_name", "variant"))

product_name,variant
SportsBar Energy Bar Choco Fudge (60g),60g
SportsBar Energy Bar Choco Fudge (40g),40g
SportsBar Energy Bar Choco Fudge (25g),25g
SportsBar Protein Bar Peanut Crunch (45g),45g
SportsBar Protein Bar Peanut Crunch (55g),55g
SportsBar Protein Bar Peanut Crunch (65g),65g
SportsBar Granola Crunch Honey Almond (400g),400g
SportsBar Granola Crunch Honey Almond (300g),300g
SportsBar Granola Crunch Honey Almond (200g),200g
SportsBar Greek Yogurt Pro Vanilla (200g),200g


### 6. Create product_code and clean product_id
`product_code`: a unique text code generated from the product name, to fit Atlikon's text-based codes.

`product_id`: non-numeric IDs are replaced with 999999 so related orders aren't lost.

In [0]:
df_silver = (
    df_silver
    .withColumn("product_code", F.sha2(F.col("product_name").cast("string"), 256))
    .withColumn(
        "product_id",
        F.when(
            F.col("product_id").cast("string").rlike("^[0-9]+$"),
            F.col("product_id").cast("string")
        ).otherwise(F.lit("999999"))
    )
    .withColumnRenamed("product_name", "product")
)

# Check: how many IDs were invalid
print("Invalid product IDs:", df_silver.filter(F.col("product_id") == "999999").count())

Invalid product IDs: 1


In [0]:
display(
    df_bronze.filter(~F.col("product_id").cast("string").rlike("^[0-9]+$"))
)

product_name,product_id,category,read_timestamp,file_name,file_size
SportsBar Oats Cookie Bites ChocoChip (350g),XYZ123,healthy snacks,2026-11-04T06:45:31.868Z,products.csv,1388


### 7. Arrange columns
Puts columns in Atlikon's order, followed by SportsBar's ID and the tracking columns.

In [0]:
df_silver = df_silver.select(
    "product_code", "division", "category", "product", "variant",
    "product_id", "read_timestamp", "file_name", "file_size"
)

display(df_silver)

product_code,division,category,product,variant,product_id,read_timestamp,file_name,file_size
e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,Nutrition Bars,Energy Bars,SportsBar Energy Bar Choco Fudge (60g),60g,25891101,2026-11-04T06:45:31.868Z,products.csv,1388
e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,Nutrition Bars,Energy Bars,SportsBar Energy Bar Choco Fudge (40g),40g,25891102,2026-11-04T06:45:31.868Z,products.csv,1388
102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,Nutrition Bars,Energy Bars,SportsBar Energy Bar Choco Fudge (25g),25g,25891103,2026-11-04T06:45:31.868Z,products.csv,1388
2e387cef1424d6e7b162b45622d4b1a788d11776e33d05cc8552f4ecd2ea1896,Nutrition Bars,Protein Bars,SportsBar Protein Bar Peanut Crunch (45g),45g,25891201,2026-11-04T06:45:31.868Z,products.csv,1388
0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28,Nutrition Bars,Protein Bars,SportsBar Protein Bar Peanut Crunch (55g),55g,25891202,2026-11-04T06:45:31.868Z,products.csv,1388
889c67757ece9c973791dfbc2d47b026a3342cc7255e47a3170329d158e897c2,Nutrition Bars,Protein Bars,SportsBar Protein Bar Peanut Crunch (65g),65g,25891203,2026-11-04T06:45:31.868Z,products.csv,1388
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (400g),400g,25891301,2026-11-04T06:45:31.868Z,products.csv,1388
d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (300g),300g,25891302,2026-11-04T06:45:31.868Z,products.csv,1388
c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (200g),200g,25891303,2026-11-04T06:45:31.868Z,products.csv,1388
da6bfc596c1360ca07bda4e0ae6bfe3b8456517fc6e8ddc265630ff940f9ab05,Dairy & Recovery,Recovery Dairy,SportsBar Greek Yogurt Pro Vanilla (200g),200g,25891401,2026-11-04T06:45:31.868Z,products.csv,1388


### Save to silver

In [0]:
(
    df_silver.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable(table_name(silver_schema, data_source))
)

## Gold

### Keep business columns and save to gold
Saves as `sb_dim_products`. Keeps `product_id` so SportsBar's orders can be linked to `product_code` later.

In [0]:
df_gold = spark.table(table_name(silver_schema, data_source)).select(
    "product_code", "product_id", "division", "category", "product", "variant"
)

(
    df_gold.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .mode("overwrite")
    .saveAsTable(table_name(gold_schema, f"sb_dim_{data_source}"))
)

display(df_gold.limit(5))

product_code,product_id,division,category,product,variant
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,25891503,Healthy Snacks,Healthy Snacks,SportsBar Oats Cookie Bites ChocoChip (180g),180g
e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,25891102,Nutrition Bars,Energy Bars,SportsBar Energy Bar Choco Fudge (40g),40g
d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5,25891302,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (300g),300g
77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,25891403,Dairy & Recovery,Recovery Dairy,SportsBar Greek Yogurt Pro Vanilla (80g),80g
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,25891301,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (400g),400g


## Merge into parent

### Prepare SportsBar products in the parent's format
Keeps only Atlikon's five columns (`product_id` stays in SportsBar's own gold table).

In [0]:
df_child_products = spark.table(table_name(gold_schema, "sb_dim_products")).select(
    "product_code", "division", "category", "product", "variant"
)

display(df_child_products.limit(5))

product_code,division,category,product,variant
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Healthy Snacks,Healthy Snacks,SportsBar Oats Cookie Bites ChocoChip (180g),180g
e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,Nutrition Bars,Energy Bars,SportsBar Energy Bar Choco Fudge (40g),40g
d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (300g),300g
77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,Dairy & Recovery,Recovery Dairy,SportsBar Greek Yogurt Pro Vanilla (80g),80g
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (400g),400g


### Check for code clashes
A matching code would overwrite an Atlikon product. Should be 0.

In [0]:
parent_codes = spark.table(table_name(gold_schema, "dim_products")).select("product_code")
clashes = parent_codes.intersect(df_child_products.select("product_code"))
print("Clashing codes:", clashes.count())

Clashing codes: 0


### Merge into Atlikon's dim_products
Updates existing products and inserts new ones, column by column. Safe to re-run.

In [0]:
target = DeltaTable.forName(spark, table_name(gold_schema, "dim_products"))

(
    target.alias("target")
    .merge(
        source=df_child_products.alias("source"),
        condition="target.product_code = source.product_code",
    )
    .whenMatchedUpdate(set={
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant",
    })
    .whenNotMatchedInsert(values={
        "product_code": "source.product_code",
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant",
    })
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

### Verify the merge
Atlikon's 397 products plus SportsBar's products should now be in one table.

In [0]:
dim_products = spark.table(table_name(gold_schema, "dim_products"))
print("Total products:", dim_products.count())
display(dim_products.groupBy("division").count().orderBy("division"))

Total products: 415


division,count
Archery,9
Badminton,18
Basketball,10
Boxing,9
Breakfast Foods,3
Camping,14
Cricket,59
Cycling,10
Dairy & Recovery,3
Fishing,10
